# Visium HD subsampling depth metrics

Two-cell pipeline:
1. **Cell 1** — load the subsampling artifact schema, pick the deepest run as reference, embed/cluster it once, and persist an analysis schema for Cell 2.
2. **Cell 2** — for every subsampled run, project it into the reference's PCA space (`scanpy.ingest`), compute clustering- and DE-stability metrics vs. the reference, optionally track a marker gene's DE signal across depth (CAR+ analysis), and write out summary CSVs + figures.

Edit the paths in the `CONFIG` block of Cell 1 (and the matching `ANALYSIS_ROOT` at the top of Cell 2) to point at your sample before running.


In [ ]:
# === Cell 1: setup, schema loader, prepare reference, persist analysis schema ===
import os, json, gc, warnings, re
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# -------------------- CONFIG (edit per sample) --------------------
SCHEMA_JSON   = "/path/to/your/sample/tenx_like_subsampling/metrics_rpc_clamped/artifact_schema.json"
ANALYSIS_ROOT = "/path/to/your/sample/tenx_like_subsampling/analysis_rpc_clamped"
FIG_DIR       = os.path.join(ANALYSIS_ROOT, "figures")

# Compute settings (stable across runs/samples)
N_PCS         = 50
N_NEI         = 20            # for neighbors & kNN Jaccard graphs later
KNN_OVERLAP_K = 15            # value used for neighbor-overlap (kNN Jaccard)
LEIDEN_RES    = 0.6
RANDOM_STATE  = 0
N_JOBS        = max(1, int(os.environ.get("OMP_NUM_THREADS", "8")))
DO_CAR_ANALYSIS = True        # toggle the optional marker-gene (e.g. CAR+) DE-vs-depth analysis in Cell 2
CAR_GENE        = "ciltacel"  # marker gene used to define the positive/negative groups (edit or ignore if DO_CAR_ANALYSIS=False)

# Try GPU UMAP if available
_USE_CUML = False
try:
    from cuml.manifold import UMAP as cuUMAP  # noqa: F401
    _USE_CUML = True
except Exception:
    _USE_CUML = False

sc.settings.verbosity = 2
warnings.filterwarnings("ignore", category=UserWarning)
os.makedirs(ANALYSIS_ROOT, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# -------------------- SCHEMA / IO --------------------
def _coerce_schema(raw: dict):
    """
    Accepts either:
      A) Artifact schema (top-level 'runs_index'):
         {
           "runs_index": {"path": "...", "columns": [...],
                          "reads_col_pattern": "...", "umis_col_pattern": "...",
                          "saturation_col_pattern": "..."},
           "metrics": {...}  # optional
         }
      B) Analysis schema variant ('inputs' / 'outputs'):
         { "inputs": { "runs_index_csv": "..." }, "outputs": {...} }
    """
    out = {}
    if isinstance(raw, dict) and "runs_index" in raw:
        ri = raw["runs_index"]
        out["runs_index_path"]      = ri["path"]
        out["runs_index_columns"]   = ri.get("columns", [])
        out["reads_col_pattern"]    = ri.get("reads_col_pattern")
        out["umis_col_pattern"]     = ri.get("umis_col_pattern")
        out["saturation_col_pattern"]= ri.get("saturation_col_pattern")
        if "metrics" in raw:
            out["metrics_path"]    = raw["metrics"].get("path")
            out["metrics_columns"] = raw["metrics"].get("columns", [])
        return out

    if isinstance(raw, dict) and "inputs" in raw and isinstance(raw["inputs"], dict):
        inp = raw["inputs"]
        if "runs_index_csv" in inp:
            out["runs_index_path"] = inp["runs_index_csv"]
        elif "runs_index" in inp:
            out["runs_index_path"] = inp["runs_index"]
        else:
            raise KeyError("Schema 'inputs' present but missing 'runs_index_csv' or 'runs_index'.")
        out["reads_col_pattern"]     = inp.get("reads_col_pattern")
        out["umis_col_pattern"]      = inp.get("umis_col_pattern")
        out["saturation_col_pattern"]= inp.get("saturation_col_pattern")
        if "outputs" in raw and isinstance(raw["outputs"], dict):
            outs = raw["outputs"]
            out["metrics_path"] = outs.get("clustering_metrics_csv")
        return out

    raise KeyError(f"Could not find runs_index path. Top-level keys: {list(raw.keys())}")

def load_schema(schema_path: str):
    with open(schema_path, "r") as f:
        raw = json.load(f)
    sch = _coerce_schema(raw)
    if not os.path.exists(sch["runs_index_path"]):
        raise FileNotFoundError(f"runs_index not found: {sch['runs_index_path']}")
    return sch

def read_runs_index(schema: dict) -> pd.DataFrame:
    """Expect columns: ['target_rpc','h5ad','p_keep'] (coerced if named slightly differently)."""
    path = schema["runs_index_path"]
    df = pd.read_csv(path)

    if "target_rpc" not in df.columns:
        cand = [c for c in df.columns if "target" in c.lower() and "rpc" in c.lower()]
        assert cand, f"No 'target_rpc' column in {path}"
        df = df.rename(columns={cand[0]: "target_rpc"})
    if "h5ad" not in df.columns:
        cand = [c for c in df.columns if "h5ad" in c.lower()]
        assert cand, f"No 'h5ad' column in {path}"
        df = df.rename(columns={cand[0]: "h5ad"})
    if "p_keep" not in df.columns:
        df["p_keep"] = np.nan

    df["target_rpc"] = pd.to_numeric(df["target_rpc"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["target_rpc", "h5ad"]).reset_index(drop=True)
    df["h5ad"] = df["h5ad"].astype(str)
    return df.sort_values("target_rpc").reset_index(drop=True)

# -------------------- small helpers --------------------
def _ensure_lognorm(ad, layer="lognorm", target_sum=1e4):
    if layer not in ad.layers:
        norm = sc.pp.normalize_total(ad, target_sum=target_sum, inplace=False)
        ad.layers[layer] = norm["X"]
        sc.pp.log1p(ad, layer=layer)

def _compute_hvgs_on_ref(ad_ref, n_top=2000, layer="lognorm"):
    sc.pp.highly_variable_genes(ad_ref, n_top_genes=n_top, flavor="seurat", layer=layer, subset=False)
    if "highly_variable" not in ad_ref.var or ad_ref.var["highly_variable"].sum() == 0:
        raise RuntimeError("Failed to compute HVGs on reference.")
    return ad_ref.var["highly_variable"].values

def _select_gene_id_column(var_df: pd.DataFrame) -> str | None:
    """
    Choose a stable feature ID column (prefer Ensembl-like) for cross-run alignment.
    Priority list; first column with unique, non-null values across all vars wins.
    """
    candidates = [
        "gene_ids", "gene_id", "feature_id", "ensembl", "ensembl_id", "ensg", "id"
    ] + [c for c in var_df.columns if c.lower().endswith("_id") or c.lower().endswith("_ids")]
    seen = set()
    ordered = []
    for c in candidates:
        cl = c if isinstance(c, str) else str(c)
        if cl in seen:
            continue
        seen.add(cl); ordered.append(cl)

    n = len(var_df)
    ens_like = re.compile(r"^ENSG\d{6,}|^ENS[A-Z]{2,}\d{6,}", re.IGNORECASE)

    # pass 1: strictly Ensembl-like if possible
    for col in ordered:
        if col in var_df.columns:
            vals = pd.Series(var_df[col]).astype(str)
            if vals.isna().any():
                continue
            if vals.nunique(dropna=True) != n:
                continue
            if (vals.str.match(ens_like)).mean() > 0.8:  # mostly Ensembl-like
                return col
    # pass 2: any unique ID column
    for col in ordered:
        if col in var_df.columns:
            vals = pd.Series(var_df[col]).astype(str)
            if vals.isna().any():
                continue
            if vals.nunique(dropna=True) == n:
                return col
    return None

def _embed_cluster_reference(ad_ref, hvgs_mask, n_pcs=N_PCS, n_neighbors=N_NEI,
                             leiden_res=LEIDEN_RES, random_state=RANDOM_STATE):
    ad_ref = ad_ref.copy()
    ad_ref.var["highly_variable"] = hvgs_mask
    sc.pp.pca(ad_ref, n_comps=n_pcs, use_highly_variable=True, layer="lognorm",
              svd_solver="arpack", random_state=random_state)
    sc.pp.neighbors(ad_ref, n_neighbors=n_neighbors, n_pcs=n_pcs, random_state=random_state)
    # UMAP mainly for visualization; ingest later will project runs into this PCA
    try:
        if _USE_CUML:
            from cuml.manifold import UMAP as cuUMAP
            emb = cuUMAP(n_neighbors=n_neighbors, n_components=2, min_dist=0.5,
                         random_state=random_state).fit_transform(ad_ref.obsm["X_pca"])
            ad_ref.obsm["X_umap"] = np.asarray(emb)
        else:
            sc.tl.umap(ad_ref, min_dist=0.5, random_state=random_state)
    except Exception:
        sc.tl.umap(ad_ref, min_dist=0.5, random_state=random_state)
    sc.tl.leiden(ad_ref, flavor="igraph", resolution=leiden_res,
                 key_added="leiden_ref", random_state=random_state)
    return ad_ref

# -------------------- LOAD SCHEMA & RUNS --------------------
schema = load_schema(SCHEMA_JSON)
runs_df = read_runs_index(schema)
print(f"[loaded] {len(runs_df)} subsample runs from index: {schema['runs_index_path']}")
print(runs_df.head())

# Choose reference as the largest target_rpc entry
ref_row = runs_df.iloc[runs_df["target_rpc"].astype(int).argmax()]
REF_H5AD = ref_row["h5ad"]
if not os.path.exists(REF_H5AD):
    raise FileNotFoundError(f"Reference h5ad not found: {REF_H5AD}")

# -------------------- PREPARE REFERENCE --------------------
ad_ref = sc.read_h5ad(REF_H5AD)
print(f"[ref] {ad_ref.n_obs:,} cells × {ad_ref.n_vars:,} genes")

# Ensure log-normalized layer exists
_ensure_lognorm(ad_ref, layer="lognorm")

# Compute HVGs on the reference (single, fixed mask used for all runs)
hvg_mask = _compute_hvgs_on_ref(ad_ref, n_top=3000, layer="lognorm")

# Choose stable gene ID column (for later alignment in ref order)
gene_id_col = _select_gene_id_column(ad_ref.var)
if gene_id_col is None:
    print("[warn] No stable gene ID column found; will fall back to gene NAMES for alignment.")
else:
    print(f"[ref] Using '{gene_id_col}' as stable feature ID column for alignment.")

# Build reference PCA/NN/UMAP/Leiden ONCE (fixed space)
ad_ref = _embed_cluster_reference(ad_ref, hvg_mask,
                                  n_pcs=N_PCS, n_neighbors=N_NEI,
                                  leiden_res=LEIDEN_RES, random_state=RANDOM_STATE)

# Reference DE (string-typed groups)
ad_ref.obs["leiden_ref"] = ad_ref.obs["leiden_ref"].astype(str)
if "de_ref" not in ad_ref.uns:
    sc.tl.rank_genes_groups(ad_ref, groupby="leiden_ref", method="wilcoxon",
                            key_added="de_ref", use_raw=False, layer="lognorm", n_genes=300)

# Save prepared reference (with PCA/UMAP/Leiden/DE)
REF_OUT = os.path.join(ANALYSIS_ROOT, "reference_with_embedding.h5ad")
ad_ref.write_h5ad(REF_OUT)
print(f"[save] reference with clustering → {REF_OUT}")

# -------------------- Persist analysis schema for Cell 2 --------------------
analysis_schema = {
    "inputs": {
        "artifact_schema_json": SCHEMA_JSON,
        "runs_index_csv": schema["runs_index_path"],
        "reference_h5ad": REF_OUT,
        "reads_col_pattern": schema.get("reads_col_pattern"),
        "umis_col_pattern": schema.get("umis_col_pattern"),
        "saturation_col_pattern": schema.get("saturation_col_pattern"),
        "gene_id_column": gene_id_col,     # may be None → fallback to var_names later
    },
    "params": {
        "n_pcs": N_PCS, "n_neighbors": N_NEI, "knn_overlap_k": KNN_OVERLAP_K,
        "leiden_res": LEIDEN_RES, "random_state": RANDOM_STATE,
        "n_jobs": N_JOBS, "use_cuml_umap": bool(_USE_CUML),
        "do_car_analysis": bool(DO_CAR_ANALYSIS), "car_gene": CAR_GENE,
    },
    "outputs": {
        "clustering_metrics_csv": os.path.join(ANALYSIS_ROOT, "clustering_stability_metrics.csv"),
        "deg_metrics_csv":        os.path.join(ANALYSIS_ROOT, "deg_stability_metrics.csv"),
        "car_deg_traces_csv":     os.path.join(ANALYSIS_ROOT, "car_deg_traces.csv"),
        "figures": {
            "stability_panel_png": os.path.join(FIG_DIR, "stability_vs_reads.png"),
            "de_metrics_panel_png":os.path.join(FIG_DIR, "de_metrics_panel.png"),
            "car_deg_traces_png":  os.path.join(FIG_DIR, "car_deg_traces.png"),
            "umap_grid_png":       os.path.join(FIG_DIR, "umap_grid.png"),
        },
    },
}
ANALYSIS_SCHEMA_JSON = os.path.join(ANALYSIS_ROOT, "analysis_artifact_schema.json")
with open(ANALYSIS_SCHEMA_JSON, "w") as f:
    json.dump(analysis_schema, f, indent=2)
print(f"[save] analysis schema → {ANALYSIS_SCHEMA_JSON}")

gc.collect()
print("[done] Cell 1 ready: reference + analysis schema written.")


In [ ]:
# === Cell 2: ingest-based depth metrics, save to H5ADs, DE robustness, optional CAR+ traces, UMAP grid ===
import os, json, gc, warnings, math, re
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse import issparse
from scipy.stats import spearmanr
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 2

# --------- locate the analysis schema produced in Cell 1 ----------
ANALYSIS_ROOT = "/path/to/your/sample/tenx_like_subsampling/analysis_rpc_clamped"  # must match Cell 1
ASCH_JSON = os.path.join(ANALYSIS_ROOT, "analysis_artifact_schema.json")
assert os.path.exists(ASCH_JSON), f"Missing analysis schema: {ASCH_JSON}"

with open(ASCH_JSON, "r") as f:
    ASCH = json.load(f)

RUNS_CSV          = ASCH["inputs"]["runs_index_csv"]
REF_H5AD          = ASCH["inputs"]["reference_h5ad"]
READS_COL_PATTERN = ASCH["inputs"]["reads_col_pattern"]
UMIS_COL_PATTERN  = ASCH["inputs"]["umis_col_pattern"]
SAT_COL_PATTERN   = ASCH["inputs"]["saturation_col_pattern"]
GENE_ID_COL       = ASCH["inputs"]["gene_id_column"]  # may be None

OUT_CLUSTERING_CSV    = ASCH["outputs"]["clustering_metrics_csv"]
OUT_DEG_CSV           = ASCH["outputs"]["deg_metrics_csv"]
OUT_CARTRACES_FC_CSV  = ASCH["outputs"]["car_deg_traces_csv"]
OUT_CARTRACES_PADJ_CSV= os.path.join(ANALYSIS_ROOT, "car_deg_traces_padj.csv")

FIG_STAB   = ASCH["outputs"]["figures"]["stability_panel_png"]
FIG_DE     = ASCH["outputs"]["figures"]["de_metrics_panel_png"]
FIG_CAR_FC = ASCH["outputs"]["figures"]["car_deg_traces_png"]
FIG_CAR_P  = os.path.join(os.path.dirname(FIG_CAR_FC), "car_deg_traces_padj.png")
FIG_GRID   = ASCH["outputs"]["figures"]["umap_grid_png"]

P              = ASCH["params"]
N_PCS          = int(P.get("n_pcs", 50))
N_NEI          = int(P.get("n_neighbors", 20))
KNN_OVERLAP_K  = int(P.get("knn_overlap_k", 15))
LEIDEN_RES     = float(P.get("leiden_res", 0.6))
RANDOM_STATE   = int(P.get("random_state", 0))
USE_CUML       = bool(P.get("use_cuml_umap", False))
DO_CAR_ANALYSIS= bool(P.get("do_car_analysis", True))
CAR_GENE       = P.get("car_gene", "ciltacel")  # marker gene defining positive/negative groups

for pth in [OUT_CLUSTERING_CSV, OUT_DEG_CSV, OUT_CARTRACES_FC_CSV, OUT_CARTRACES_PADJ_CSV,
            FIG_STAB, FIG_DE, FIG_CAR_FC, FIG_CAR_P, FIG_GRID]:
    os.makedirs(os.path.dirname(pth), exist_ok=True)

# ---------------- helpers ----------------
def _ensure_lognorm(ad, layer="lognorm", target_sum=1e4):
    if layer not in ad.layers:
        norm = sc.pp.normalize_total(ad, target_sum=target_sum, inplace=False)
        ad.layers[layer] = norm["X"]
        sc.pp.log1p(ad, layer=layer)

def _save_basic_qc_to_obs(ad):
    X = ad.X
    if issparse(X):
        n_counts = np.asarray(X.sum(axis=1)).ravel()
        n_genes  = np.asarray((X > 0).sum(axis=1)).ravel()
    else:
        n_counts = X.sum(axis=1).ravel()
        n_genes  = (X > 0).sum(axis=1).ravel()
    ad.obs["total_counts"] = n_counts
    ad.obs["n_genes_by_counts"] = n_genes

def _build_ref_id_index(ad_ref, gene_id_col: str | None):
    """Return Series: index=ref_var_index (0..), values=stable_id strings (fallback to var_names)."""
    if gene_id_col and gene_id_col in ad_ref.var.columns:
        vals = pd.Series(ad_ref.var[gene_id_col]).astype(str)
        if (vals.isna().any()) or (vals.nunique() != len(vals)):
            gene_id_col = None
    if not gene_id_col:
        vals = pd.Series(ad_ref.var_names).astype(str)
    return vals.reset_index(drop=True)

def _align_by_gene_ids(ad_q, ad_ref, gene_id_col: str | None):
    """
    Align query to reference variable order using stable IDs (prefer Ensembl-like).
    Returns a *new* AnnData with columns in reference order (intersection only).
    """
    ref_ids = _build_ref_id_index(ad_ref, gene_id_col)
    if gene_id_col and gene_id_col in ad_q.var.columns:
        q_ids = pd.Series(ad_q.var[gene_id_col]).astype(str).reset_index(drop=True)
    else:
        q_ids = pd.Series(ad_q.var_names).astype(str).reset_index(drop=True)

    # map each ref id to first matching query index
    q_lut = pd.Series(q_ids.index.values, index=q_ids.values)
    take = []
    for rid in ref_ids:
        if rid in q_lut.index:
            j = int(q_lut.loc[rid])
            take.append(j)
        else:
            take.append(None)

    mask = np.array([j is not None for j in take], dtype=bool)
    if not mask.any():
        raise RuntimeError("No overlapping genes between run and reference after ID alignment.")
    q_idx = np.array([j for j in take if j is not None], dtype=int)
    ref_sub = ad_ref[:, mask].copy()
    q_sub   = ad_q[:, q_idx].copy()
    # keep same var order and copy over ref HVG mask if present
    if "highly_variable" in ad_ref.var.columns:
        ref_hvg = ad_ref.var["highly_variable"].values[mask]
        q_sub.var["highly_variable"] = ref_hvg
    return ref_sub, q_sub, mask, q_idx

def _project_into_ref_space(ad_q, ad_ref, n_neighbors=N_NEI, n_pcs=N_PCS, leiden_res=LEIDEN_RES):
    """
    Project ad_q into ad_ref PCA space via scanpy.ingest on *float lognorm*.
    Then rebuild neighbors on X_pca, compute UMAP & Leiden (key: 'leiden_ds').
    """
    X_backup = ad_q.X
    try:
        X_ln = ad_q.layers["lognorm"]
        if sparse.issparse(X_ln): X_ln = X_ln.astype(np.float32)
        else: X_ln = X_ln.astype(np.float32, copy=False)
        ad_q.X = X_ln
        sc.tl.ingest(ad_q, ad_ref, obs=None)  # writes X_pca etc.
    finally:
        ad_q.X = X_backup

    sc.pp.neighbors(ad_q, n_neighbors=n_neighbors, n_pcs=n_pcs, use_rep="X_pca", random_state=RANDOM_STATE)
    try:
        if USE_CUML:
            from cuml.manifold import UMAP as cuUMAP
            emb = cuUMAP(n_neighbors=n_neighbors, n_components=2, min_dist=0.5,
                         random_state=RANDOM_STATE).fit_transform(ad_q.obsm["X_pca"])
            ad_q.obsm["X_umap"] = np.asarray(emb)
        else:
            sc.tl.umap(ad_q, min_dist=0.5, random_state=RANDOM_STATE)
    except Exception:
        sc.tl.umap(ad_q, min_dist=0.5, random_state=RANDOM_STATE)

    sc.tl.leiden(ad_q, flavor="igraph", resolution=leiden_res, key_added="leiden_ds", random_state=RANDOM_STATE)
    # keep a convenience alias too
    ad_q.obs["leiden"] = ad_q.obs["leiden_ds"].astype(str).values
    return ad_q

def _neighbor_overlap_jaccard(A_ref, A_q, k=KNN_OVERLAP_K) -> float:
    """Average Jaccard of top-k neighbors per cell between two CSR graphs with same cell order."""
    if not sparse.isspmatrix_csr(A_ref): A_ref = A_ref.tocsr()
    if not sparse.isspmatrix_csr(A_q):   A_q   = A_q.tocsr()
    n = A_ref.shape[0]
    tot = 0.0; cnt = 0
    for i in range(n):
        sr = slice(A_ref.indptr[i], A_ref.indptr[i+1])
        idx_r = A_ref.indices[sr]; dat_r = A_ref.data[sr]
        if idx_r.size == 0: continue
        kr = min(k, idx_r.size)
        top_r = idx_r[np.argpartition(-dat_r, kr-1)[:kr]]

        sd = slice(A_q.indptr[i], A_q.indptr[i+1])
        idx_d = A_q.indices[sd]; dat_d = A_q.data[sd]
        if idx_d.size == 0: continue
        kd = min(k, idx_d.size)
        top_d = idx_d[np.argpartition(-dat_d, kd-1)[:kd]]

        s1, s2 = set(top_r), set(top_d)
        u = len(s1 | s2)
        if u:
            tot += len(s1 & s2)/u; cnt += 1
    return float(tot/cnt) if cnt else np.nan

def _hungarian_match(y_true, y_pred):
    tcat = pd.Categorical(y_true); pcat = pd.Categorical(y_pred)
    cm = np.zeros((len(tcat.categories), len(pcat.categories)), dtype=int)
    for t, p in zip(tcat.codes, pcat.codes):
        if t >= 0 and p >= 0: cm[t, p] += 1
    r, c = linear_sum_assignment(-cm)
    mapping = {pcat.categories[cj]: tcat.categories[ri] for ri, cj in zip(r, c)}
    acc = cm[r, c].sum() / cm.sum() if cm.sum() > 0 else np.nan
    return mapping, float(acc)

def _deg_table(ad, key, group):
    try:
        return sc.get.rank_genes_groups_df(ad, key=key, group=group)
    except Exception:
        rg = ad.uns[key]
        names = pd.Series(rg["names"][group], name="names")
        pvals_adj = pd.Series(rg["pvals_adj"][group], name="pvals_adj")
        logfc = pd.Series(rg["logfoldchanges"][group], name="logfoldchanges")
        return pd.DataFrame({"names": names, "pvals_adj": pvals_adj, "logfoldchanges": logfc})

def _topN_jaccard(list_a, list_b, N=20):
    if len(list_a)==0 and len(list_b)==0: return np.nan
    a = set(list_a[:N]); b = set(list_b[:N])
    u = len(a|b)
    return len(a&b)/u if u else np.nan

def _total_reads_for_run(ad, target_rpc):
    if READS_COL_PATTERN:
        col = READS_COL_PATTERN.format(target_rpc=int(target_rpc))
        if col in ad.obs.columns:
            return float(np.asarray(ad.obs[col]).sum())
    cand = [c for c in ad.obs.columns if str(c).startswith("reads_sim_")]
    if cand:
        return float(np.asarray(ad.obs[cand[0]]).sum())
    return float(np.asarray(ad.X.sum(axis=1)).sum())

def _find_gene_index(ad, gene_name):
    if gene_name in ad.var_names: return ad.var_names.get_loc(gene_name)
    up = ad.var_names.str.upper() == gene_name.upper()
    if up.any(): return int(np.where(up)[0][0])
    return None

def _log2fc_for_genes(ad, pos_mask, genes, layer="lognorm", eps=1e-9):
    X = ad.layers[layer] if layer in ad.layers else ad.X
    X = X.toarray() if issparse(X) else np.asarray(X)
    pos = X[pos_mask]; neg = X[~pos_mask]
    gidx = [ad.var_names.get_loc(g) if g in ad.var_names else None for g in genes]
    out = []
    for g, j in zip(genes, gidx):
        if j is None: out.append(np.nan); continue
        mu_pos = float(np.mean(pos[:, j])); mu_neg = float(np.mean(neg[:, j]))
        out.append(np.log2((mu_pos + eps) / (mu_neg + eps)))
    return out

# ---------------- load reference & runs ----------------
runs = pd.read_csv(RUNS_CSV).sort_values("target_rpc").reset_index(drop=True)
ad_ref = sc.read_h5ad(REF_H5AD)
_ensure_lognorm(ad_ref, layer="lognorm")

# metrics collectors
metrics_rows, deg_rows = [], []

print(f"[ref] {ad_ref.n_obs:,} cells × {ad_ref.n_vars:,} genes | GENE_ID_COL={GENE_ID_COL}")
print("[start] per-run projection into reference space via ingest")
for _, r in runs.iterrows():
    rpc = int(r["target_rpc"]); h5ad_path = str(r["h5ad"])
    if not os.path.exists(h5ad_path):
        print(f"[skip] missing {h5ad_path}")
        continue

    print(f"\n[run] target_rpc={rpc} → {os.path.basename(h5ad_path)}")
    ad = sc.read_h5ad(h5ad_path)
    _ensure_lognorm(ad, layer="lognorm")
    _save_basic_qc_to_obs(ad)

    # --- align genes by stable IDs to reference order (intersection only) ---
    ref_al, ad_al, ref_mask, ad_idx = _align_by_gene_ids(ad, ad_ref, GENE_ID_COL)

    # --- project (ingest) into reference PCA space; then neighbors/UMAP/Leiden ---
    ad_al = _project_into_ref_space(ad_al, ad_ref, n_neighbors=N_NEI, n_pcs=N_PCS, leiden_res=LEIDEN_RES)

    # ---- DE on aligned object (for fair comparison to ref DE) ----
    sc.tl.rank_genes_groups(ad_al, groupby="leiden_ds", method="wilcoxon",
                            key_added="de_aligned", use_raw=False, layer="lognorm", n_genes=300)

    # --- Clustering comparatives (same cells/order) ---
    y_true = ref_al.obs["leiden_ref"].astype(str).values
    y_pred = ad_al.obs["leiden_ds"].astype(str).values
    ARI = adjusted_rand_score(y_true, y_pred)
    NMI = normalized_mutual_info_score(y_true, y_pred)
    mapping, map_acc = _hungarian_match(y_true, y_pred)
    nnJ = _neighbor_overlap_jaccard(ref_al.obsp["connectivities"], ad_al.obsp["connectivities"], k=KNN_OVERLAP_K)

    # --- DE robustness (mean across matched clusters) ---
    jcats, rhos = [], []
    for qlab, tlab in mapping.items():
        df_ref = _deg_table(ref_al, key="de_ref", group=tlab)
        df_q   = _deg_table(ad_al,  key="de_aligned", group=qlab)
        # overlap on adj pval ranking
        jcats.append(_topN_jaccard(
            df_ref.sort_values("pvals_adj")["names"].tolist(),
            df_q.sort_values("pvals_adj")["names"].tolist(), N=20
        ))
        # logFC rho on intersection
        m = df_ref[["names","logfoldchanges"]].merge(
            df_q[["names","logfoldchanges"]], on="names",
            suffixes=("_ref","_q"))
        if len(m) >= 10:
            rhos.append(spearmanr(m["logfoldchanges_ref"], m["logfoldchanges_q"]).correlation)

        # store per-cluster jaccard (drill-down CSV)
        deg_rows.append(dict(
            target_rpc=rpc, cluster_ref=str(tlab), cluster_q=str(qlab),
            top20_jaccard=_topN_jaccard(
                df_ref.sort_values("pvals_adj")["names"].tolist(),
                df_q.sort_values("pvals_adj")["names"].tolist(), N=20
            ),
            h5ad=h5ad_path
        ))

    de_top20 = float(np.nanmean(jcats)) if jcats else np.nan
    de_rho   = float(np.nanmean(rhos))  if rhos else np.nan

    # --- totals & meta ---
    total_reads = _total_reads_for_run(ad, rpc)
    reads_m = total_reads / 1e6
    metrics_rows.append(dict(
        target_rpc=rpc, total_reads=total_reads, reads_millions=reads_m,
        ARI=ARI, NMI=NMI, cluster_map_acc=map_acc, neighbor_jaccard=nnJ,
        de_top20_jaccard=de_top20, de_logfc_spearman=de_rho,
        h5ad=h5ad_path
    ))

    # --- persist back to original H5AD (store embeddings/graph/labels & QC) ---
    ad.obsm["X_pca_refspace"] = ad_al.obsm["X_pca"]
    ad.obsm["X_umap_refspace"] = ad_al.obsm.get("X_umap", None)
    ad.obsp["connectivities_refspace"] = ad_al.obsp["connectivities"]
    if "distances" in ad_al.obsp:  # sometimes distances not kept; guard
        ad.obsp["distances_refspace"] = ad_al.obsp["distances"]
    # Save labels: 'leiden_ds' and convenience alias 'leiden'
    ad.obs["leiden_ds"] = pd.Categorical(ad_al.obs["leiden_ds"].astype(str).reindex(ad.obs_names).fillna("NA"))
    ad.obs["leiden"]    = ad.obs["leiden_ds"].astype(str)

    # Keep basic QC if not present
    if "total_counts" not in ad.obs: _save_basic_qc_to_obs(ad)

    # Save summary metrics for this run
    ad.uns.setdefault("subsample_metrics", {})
    ad.uns["subsample_metrics"].update(dict(
        target_rpc=rpc, reads_millions=reads_m,
        ARI=ARI, NMI=NMI, cluster_map_acc=map_acc, neighbor_jaccard=nnJ,
        de_top20_jaccard=de_top20, de_logfc_spearman=de_rho,
        params=dict(n_pcs=N_PCS, n_neighbors=N_NEI, knn_overlap_k=KNN_OVERLAP_K,
                    leiden_res=LEIDEN_RES, random_state=RANDOM_STATE, use_cuml_umap=USE_CUML)
    ))

    # Also keep DE table computed on aligned object
    try:
        ad.uns["de_aligned"] = ad_al.uns["de_aligned"]
    except Exception:
        pass

    # Write back
    ad.write_h5ad(h5ad_path)
    print(f"[save] updated embeddings/labels/metrics → {h5ad_path}")

    del ad, ad_al, ref_al
    gc.collect()

# ---------- Save metric CSVs ----------
metrics_df = pd.DataFrame(metrics_rows).sort_values("reads_millions")
deg_df     = pd.DataFrame(deg_rows).sort_values(["target_rpc","cluster_ref"])
metrics_df.to_csv(OUT_CLUSTERING_CSV, index=False)
deg_df.to_csv(OUT_DEG_CSV, index=False)
print(f"\n[save] clustering metrics → {OUT_CLUSTERING_CSV}")
print(f"[save] per-cluster DE metrics → {OUT_DEG_CSV}")

# ---------------- Figures: clustering & DE panels ----------------
if not metrics_df.empty:
    X = metrics_df["reads_millions"].to_numpy()

    # Stability panel (6 panels)
    fig, axes = plt.subplots(3, 2, figsize=(12, 12), constrained_layout=True)
    axes = axes.ravel()
    axes[0].plot(X, metrics_df["ARI"], marker="o"); axes[0].set_title("ARI vs total reads"); axes[0].set_xlabel("Reads (M)"); axes[0].grid(True, alpha=0.3)
    axes[1].plot(X, metrics_df["NMI"], marker="o"); axes[1].set_title("NMI vs total reads"); axes[1].set_xlabel("Reads (M)"); axes[1].grid(True, alpha=0.3)
    axes[2].plot(X, metrics_df["cluster_map_acc"], marker="o"); axes[2].set_title("Cluster map accuracy (Hungarian)"); axes[2].set_xlabel("Reads (M)"); axes[2].grid(True, alpha=0.3)
    axes[3].plot(X, metrics_df["neighbor_jaccard"], marker="o"); axes[3].set_title(f"kNN Jaccard (k={KNN_OVERLAP_K})"); axes[3].set_xlabel("Reads (M)"); axes[3].grid(True, alpha=0.3)
    axes[4].plot(X, metrics_df["de_top20_jaccard"], marker="o"); axes[4].set_title("DE Top-20 Jaccard"); axes[4].set_xlabel("Reads (M)"); axes[4].grid(True, alpha=0.3)
    axes[5].plot(X, metrics_df["de_logfc_spearman"], marker="o"); axes[5].set_title("DE log2FC Spearman"); axes[5].set_xlabel("Reads (M)"); axes[5].grid(True, alpha=0.3)
    plt.savefig(FIG_STAB, dpi=160, bbox_inches="tight"); plt.close()
    print(f"[save] stability panel → {FIG_STAB}")

    # DE-only compact panel
    plt.figure(figsize=(10, 4))
    plt.subplot(1,2,1)
    plt.plot(X, metrics_df["de_top20_jaccard"], marker="o")
    plt.title("DE Top-20 Jaccard"); plt.xlabel("Reads (M)"); plt.grid(True, alpha=0.3)
    plt.subplot(1,2,2)
    plt.plot(X, metrics_df["de_logfc_spearman"], marker="o")
    plt.title("DE log2FC Spearman"); plt.xlabel("Reads (M)"); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DE, dpi=160, bbox_inches="tight"); plt.close()
    print(f"[save] DE metrics panel → {FIG_DE}")

# ---------------- Optional: marker-gene positive vs negative DE trajectory (e.g. CAR+) ----------------
if DO_CAR_ANALYSIS:
    if not metrics_df.empty:
        # deepest run by reads → derive the top-10 marker-associated genes (excluding the marker itself)
        runs_df = pd.read_csv(RUNS_CSV).sort_values("target_rpc")
        rpc_max = int(metrics_df.iloc[metrics_df["reads_millions"].argmax()]["target_rpc"])
        h5_max  = runs_df.set_index("target_rpc").loc[rpc_max, "h5ad"]

        ad_max = sc.read_h5ad(h5_max)
        _ensure_lognorm(ad_max, layer="lognorm")

        j_car = _find_gene_index(ad_max, CAR_GENE)
        if j_car is not None:
            Xr = ad_max.X.toarray() if issparse(ad_max.X) else np.asarray(ad_max.X)
            car_pos_mask = Xr[:, j_car] > 0

            ad_max_tmp = ad_max.copy()
            ad_max_tmp.obs["__bin__"] = np.where(car_pos_mask, "MARKERpos", "MARKERneg")
            sc.tl.rank_genes_groups(ad_max_tmp, groupby="__bin__", groups=["MARKERpos"], reference="MARKERneg",
                                    method="wilcoxon", use_raw=False, layer="lognorm",
                                    key_added="de_markermax", n_genes=1000)
            df_max = sc.get.rank_genes_groups_df(ad_max_tmp, key="de_markermax", group="MARKERpos").sort_values("pvals_adj")
            # exclude the marker gene itself
            TOP10 = [g for g in df_max["names"].tolist() if g != CAR_GENE][:10]
            print(f"[marker] top10 (excluding {CAR_GENE}): {TOP10}")

            # sweep all runs → log2FC & padj traces
            fc_rows, padj_rows = [], []
            for _, rr in runs_df.iterrows():
                rpc = int(rr["target_rpc"]); h5ad = str(rr["h5ad"])
                if not os.path.exists(h5ad): continue
                ad = sc.read_h5ad(h5ad)
                _ensure_lognorm(ad, layer="lognorm")

                j_local = _find_gene_index(ad, CAR_GENE)
                if j_local is None:
                    continue
                Xl = ad.X.toarray() if issparse(ad.X) else np.asarray(ad.X)
                car_mask = Xl[:, j_local] > 0

                # per-run DE for marker+ vs marker-
                ad_tmp = ad.copy()
                ad_tmp.obs["__bin__"] = np.where(car_mask, "MARKERpos", "MARKERneg")
                sc.tl.rank_genes_groups(ad_tmp, groupby="__bin__", groups=["MARKERpos"], reference="MARKERneg",
                                        method="wilcoxon", use_raw=False, layer="lognorm",
                                        key_added="de_marker", n_genes=1000)
                df_run = sc.get.rank_genes_groups_df(ad_tmp, key="de_marker", group="MARKERpos")

                # reads
                total_reads = _total_reads_for_run(ad, rpc)
                reads_m = total_reads / 1e6

                # log2FC traces
                log2fcs = _log2fc_for_genes(ad, car_mask, TOP10, layer="lognorm")
                for g, l2 in zip(TOP10, log2fcs):
                    fc_rows.append(dict(target_rpc=rpc, reads_millions=reads_m, gene=g, log2FC=l2))

                # padj traces
                padj_map = dict(zip(df_run["names"], df_run["pvals_adj"]))
                for g in TOP10:
                    padj = float(padj_map.get(g, np.nan))
                    neglog10 = -np.log10(padj) if (padj is not None and padj > 0) else np.nan
                    padj_rows.append(dict(target_rpc=rpc, reads_millions=reads_m, gene=g,
                                          padj=padj, neglog10_padj=neglog10))

                del ad_tmp, ad
                gc.collect()

            traces_fc_df   = pd.DataFrame(fc_rows).sort_values(["gene","reads_millions"])
            traces_padj_df = pd.DataFrame(padj_rows).sort_values(["gene","reads_millions"])
            traces_fc_df.to_csv(OUT_CARTRACES_FC_CSV, index=False)
            traces_padj_df.to_csv(OUT_CARTRACES_PADJ_CSV, index=False)
            print(f"[save] marker+ log2FC trajectories → {OUT_CARTRACES_FC_CSV}")
            print(f"[save] marker+ padj trajectories → {OUT_CARTRACES_PADJ_CSV}")

            # plots
            plt.figure(figsize=(12, 7))
            for g, df_g in traces_fc_df.groupby("gene"):
                plt.plot(df_g["reads_millions"], df_g["log2FC"], marker="o", label=g)
            plt.axhline(0, linestyle="--", linewidth=1)
            plt.xlabel("Total reads (millions)"); plt.ylabel(f"log2FC (marker+ vs marker−), lognorm")
            plt.title(f"Top-10 marker+ DE genes (excl. {CAR_GENE}): log2FC vs reads")
            plt.legend(ncol=2, fontsize=8); plt.grid(True, alpha=0.3)
            plt.savefig(FIG_CAR_FC, dpi=160, bbox_inches="tight"); plt.close()
            print(f"[save] marker log2FC traces → {FIG_CAR_FC}")

            plt.figure(figsize=(12, 7))
            for g, df_g in traces_padj_df.groupby("gene"):
                plt.plot(df_g["reads_millions"], df_g["neglog10_padj"], marker="o", label=g)
            plt.xlabel("Total reads (millions)"); plt.ylabel("-log10 adjusted p-value")
            plt.title(f"Top-10 marker+ DE genes (excl. {CAR_GENE}): significance vs reads")
            plt.legend(ncol=2, fontsize=8); plt.grid(True, alpha=0.3)
            plt.savefig(FIG_CAR_P, dpi=160, bbox_inches="tight"); plt.close()
            print(f"[save] marker padj traces → {FIG_CAR_P}")
        else:
            print(f"[marker][warn] Gene '{CAR_GENE}' not found in deepest run; skipping marker-gene analysis.")
    else:
        print("[marker] metrics_df empty; skipping marker-gene analysis.")
else:
    print("[marker] DO_CAR_ANALYSIS=False → skipping marker-gene analysis.")

# ---------------- UMAP grid over all runs (clustered in refspace) ----------------
paths = pd.read_csv(RUNS_CSV).sort_values("target_rpc")["h5ad"].tolist()
n = len(paths)
if n > 0:
    ncols = int(math.ceil(math.sqrt(n)))
    nrows = int(math.ceil(n / ncols))

    mins = np.array([np.inf, np.inf]); maxs = np.array([-np.inf, -np.inf])
    cache = []
    for p in paths:
        if not os.path.exists(p):
            cache.append((None, None, None)); continue
        ad = sc.read_h5ad(p)
        # use the refspace embedding if available; fallback to any X_umap; else compute quickly
        if "X_umap_refspace" in ad.obsm_keys():
            Xumap = np.asarray(ad.obsm["X_umap_refspace"])
        elif "X_umap" in ad.obsm_keys():
            Xumap = np.asarray(ad.obsm["X_umap"])
        else:
            _ensure_lognorm(ad, "lognorm")
            sc.pp.pca(ad, n_comps=20, layer="lognorm")
            sc.pp.neighbors(ad, n_neighbors=15, n_pcs=20)
            sc.tl.umap(ad, random_state=RANDOM_STATE)
            Xumap = np.asarray(ad.obsm["X_umap"])
            ad.write_h5ad(p)  # cache
        labs = ad.obs["leiden_ds"].astype(str).values if "leiden_ds" in ad.obs else (
               ad.obs["leiden"].astype(str).values if "leiden" in ad.obs else None)
        cache.append((Xumap, labs, os.path.basename(p)))
        mins = np.minimum(mins, Xumap.min(axis=0)); maxs = np.maximum(maxs, Xumap.max(axis=0))
        del ad; gc.collect()

    fig, axes = plt.subplots(nrows, ncols, figsize=(3.8*ncols, 3.8*nrows), squeeze=False)
    axlist = axes.ravel()
    for i, (Xumap, labs, title) in enumerate(cache):
        ax = axlist[i]
        if Xumap is None: ax.axis("off"); continue
        if labs is None:
            ax.scatter(Xumap[:,0], Xumap[:,1], s=1, alpha=0.7)
        else:
            for lab in np.unique(labs):
                m = (labs == lab)
                ax.scatter(Xumap[m,0], Xumap[m,1], s=1, alpha=0.7)
        ax.set_title(title, fontsize=9)
        ax.set_xlim(mins[0], maxs[0]); ax.set_ylim(mins[1], maxs[1])
        ax.set_xticks([]); ax.set_yticks([])
    for j in range(len(cache), len(axlist)):
        axlist[j].axis("off")
    plt.tight_layout()
    plt.savefig(FIG_GRID, dpi=160, bbox_inches="tight"); plt.close()
    print(f"[save] UMAP grid → {FIG_GRID}")

print("[done] Cell 2 complete.")
